# 21 · Group statistics and publication figures

Subject results remain independent; group summaries use only subjects with 20-mm coverage. With n=2 these are descriptive/exploratory, not confirmatory population inference.

In [ ]:
from pathlib import Path
import sys,itertools,numpy as np,pandas as pd
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'; sys.path.insert(0,str(PIPE)); import config
from utils.plotting_science import group_decoding,effect_forest,set_science_style
group=config.BATCH_RESULT_ROOT/'group'; cov=pd.read_csv(group/'tables/coverage_summary.csv'); subjects=cov.loc[cov.n_candidates_20mm>0,'subject'].tolist(); rows=[]
for subject in subjects:
  for modality in ('erp','hg'):
    p=config.batch_subject_dir(subject)/'task1'/f'{modality}_channel_effects.csv'; d=pd.read_csv(p); rows.append({'subject':subject,'modality':modality,'effect':d.effect.mean(),'se':np.sqrt(np.square(d.se).mean()/len(d)),'n_channels':len(d)})
effects=pd.DataFrame(rows); effects.to_csv(group/'tables'/'task1_group_effects.csv',index=False); display(effects)

In [ ]:
for modality in ('erp','hg'):
  for analysis in ('task2','task3','cross_task'):
    curves=[]; times=None
    for subject in subjects:
      d=pd.read_csv(config.batch_subject_dir(subject)/analysis/f'{modality}_decoding.csv'); curves.append(d.accuracy.to_numpy()); times=d.time_ms.to_numpy()
    curves=np.asarray(curves); group_decoding(times,curves,subjects,f'Group · {analysis} · {modality.upper()}',group/'figures'/f'{analysis}_{modality}_group')
    mean=curves.mean(0); deviations=curves-.5; null=[]
    for signs in itertools.product([-1,1],repeat=len(curves)): null.append(.5+np.mean(deviations*np.asarray(signs)[:,None],axis=0))
    null=np.asarray(null); p=(1+(null>=mean).sum(0))/(1+len(null)); pd.DataFrame({'time_ms':times,'mean_accuracy':mean,'p_signflip':p,'n_subjects':len(subjects)}).to_csv(group/'tables'/f'{analysis}_{modality}_group_curve.csv',index=False)

In [ ]:
set_science_style(); print('Group figures/tables written:',group)